# Oracle AI Vector Search 结合文档处理
Oracle AI Vector Search 专为人工智能 (AI) 工作负载设计，允许您基于语义而非关键词来查询数据。
Oracle AI Vector Search 的最大优势之一是，可以在一个系统中将非结构化数据的语义搜索与业务数据的关系搜索相结合。
这不仅功能强大，而且效率显著更高，因为您无需添加专门的向量数据库，从而消除了多系统间数据碎片化的痛点。

此外，您的向量还可以受益于 Oracle Database 所有最强大的功能，例如：

 * [分区支持](https://www.oracle.com/database/technologies/partitioning.html)
 * [Real Application Clusters 可扩展性](https://www.oracle.com/database/real-application-clusters/)
 * [Exadata 智能扫描](https://www.oracle.com/database/technologies/exadata/software/smartscan/)
 * [跨地理分布式数据库的分片处理](https://www.oracle.com/database/distributed-database/)
 * [事务](https://docs.oracle.com/en/database/oracle/oracle-database/23/cncpt/transactions.html)
 * [并行 SQL](https://docs.oracle.com/en/database/oracle/oracle-database/21/vldbg/parallel-exec-intro.html#GUID-D28717E4-0F77-44F5-BB4E-234C31D4E4BA)
 * [灾难恢复](https://www.oracle.com/database/data-guard/)
 * [安全性](https://www.oracle.com/security/database-security/)
 * [Oracle Machine Learning](https://www.oracle.com/artificial-intelligence/database-machine-learning/)
 * [Oracle Graph Database](https://www.oracle.com/database/integrated-graph-database/)
 * [Oracle Spatial and Graph](https://www.oracle.com/database/spatial/)
 * [Oracle Blockchain](https://docs.oracle.com/en/database/oracle/oracle-database/23/arpls/dbms_blockchain_table.html#GUID-B469E277-978E-4378-A8C1-26D3FF96C9A6)
 * [JSON](https://docs.oracle.com/en/database/oracle/oracle-database/23/adjsn/json-in-oracle-database.html)

本指南演示了如何将 Oracle AI Vector Search 与 LangChain 结合使用，以服务于端到端的 RAG 管道。本指南将通过以下示例进行说明：

 * 使用 OracleDocLoader 从各种来源加载文档
 * 在数据库内/外使用 OracleSummary 对其进行摘要
 * 在数据库内/外使用 OracleEmbeddings 为其生成嵌入
 * 使用 OracleTextSplitter 的高级 Oracle 功能根据不同需求对其进行分块
 * 将其存储和索引到向量存储中，并针对 OracleVS 中的查询进行查询

如果您刚开始接触 Oracle Database，可以考虑探索[免费的 Oracle 23 AI](https://www.oracle.com/database/free/#resources)，它能很好地介绍如何设置您的数据库环境。在使用数据库时，通常建议默认不要使用 system 用户；相反，您可以创建自己的用户以增强安全性和自定义。有关用户创建的详细步骤，请参阅我们的[端到端指南](https://github.com/langchain-ai/langchain/blob/master/cookbook/oracleai_demo.ipynb)，其中还展示了如何在 Oracle 中设置用户。此外，了解用户权限对于有效管理数据库安全至关重要。您可以在官方的[Oracle 用户账户和安全管理指南](https://docs.oracle.com/en/database/oracle/oracle-database/19/admqs/administering-user-accounts-and-security.html#GUID-36B21D72-1BBB-46C9-A0C9-F0D2A8591B8D)中了解更多关于此主题的信息。

### 先决条件

请安装 Oracle Database [python-oracledb 驱动程序](https://pypi.org/project/oracledb/)，以便将 LangChain 与 Oracle AI Vector Search 结合使用：

```
$ python -m pip install --upgrade oracledb

### 创建演示用户
首先，以特权用户身份连接，创建一个具有所有必需权限的演示用户。请更改您环境的凭据。同时，将 `DEMO_PY_DIR` 路径设置为模型文件所在数据库主机上的目录：

In [ ]:
import oracledb

# Please update with your SYSTEM (or privileged user) username, password, and database connection string
username = "SYSTEM"
password = ""
dsn = ""

with oracledb.connect(user=username, password=password, dsn=dsn) as connection:
    print("Connection successful!")

    with connection.cursor() as cursor:
        cursor.execute(
            """
            begin
                -- Drop user
                execute immediate 'drop user if exists testuser cascade';

                -- Create user and grant privileges
                execute immediate 'create user testuser identified by testuser';
                execute immediate 'grant connect, unlimited tablespace, create credential, create procedure, create any index to testuser';
                execute immediate 'create or replace directory DEMO_PY_DIR as ''/home/yourname/demo/orachain''';
                execute immediate 'grant read, write on directory DEMO_PY_DIR to public';
                execute immediate 'grant create mining model to testuser';

                -- Network access
                begin
                    DBMS_NETWORK_ACL_ADMIN.APPEND_HOST_ACE(
                        host => '*',
                        ace => xs$ace_type(privilege_list => xs$name_list('connect'),
                                           principal_name => 'testuser',
                                           principal_type => xs_acl.ptype_db)
                    );
                end;
            end;
            """
        )
        print("User setup done!")

## 使用 Oracle AI 处理文档
请考虑以下场景：用户拥有存储在 Oracle 数据库或文件系统中的文档，并打算将这些数据与由 LangChain 驱动的 Oracle AI Vector Search 一起使用。

为了准备数据进行分析，需要一个全面的预处理工作流。首先，必须检索文档，（如果需要）进行摘要，并根据需要进行分块。后续步骤包括为这些块生成嵌入，并将它们集成到 Oracle AI Vector Store 中。然后，用户可以对此数据进行语义搜索。

Oracle AI Vector Search LangChain 库包含一套文档处理工具，可促进文档加载、分块、摘要生成和嵌入创建。

在接下来的章节中，我们将详细介绍如何利用 Oracle AI LangChain API 有效地实现这些过程。

### 连接到演示用户
以下示例代码展示了如何使用 python-oracledb 驱动程序连接到 Oracle Database。默认情况下，python-oracledb 以“Thin”模式运行，该模式直接连接到 Oracle Database。此模式不需要 Oracle Client 库。但是，当 python-oracledb 使用这些库时，一些附加功能将可用。当使用 Oracle Client 库时，python-oracledb 被称为处于“Thick”模式。两种模式都支持 Python Database API v2.0 规范的全面功能。请参阅以下[指南](https://python-oracledb.readthedocs.io/en/latest/user_guide/appendix_a.html#featuresummary)，其中介绍了每种模式支持的功能。如果无法使用 Thin 模式，您可以切换到 Thick 模式。

In [ ]:
import oracledb

# please update with your username, password, and database connection string
username = "testuser"
password = ""
dsn = ""

connection = oracledb.connect(user=username, password=password, dsn=dsn)
print("Connection successful!")

### 填充演示表
创建一个演示表并插入一些示例文档。

In [ ]:
with connection.cursor() as cursor:
    drop_table_sql = """drop table if exists demo_tab"""
    cursor.execute(drop_table_sql)

    create_table_sql = """create table demo_tab (id number, data clob)"""
    cursor.execute(create_table_sql)

    insert_row_sql = """insert into demo_tab values (:1, :2)"""
    rows_to_insert = [
        (
            1,
            "If the answer to any preceding questions is yes, then the database stops the search and allocates space from the specified tablespace; otherwise, space is allocated from the database default shared temporary tablespace.",
        ),
        (
            2,
            "A tablespace can be online (accessible) or offline (not accessible) whenever the database is open.\nA tablespace is usually online so that its data is available to users. The SYSTEM tablespace and temporary tablespaces cannot be taken offline.",
        ),
        (
            3,
            "The database stores LOBs differently from other data types. Creating a LOB column implicitly creates a LOB segment and a LOB index. The tablespace containing the LOB segment and LOB index, which are always stored together, may be different from the tablespace containing the table.\nSometimes the database can store small amounts of LOB data in the table itself rather than in a separate LOB segment.",
        ),
    ]
    cursor.executemany(insert_row_sql, rows_to_insert)

connection.commit()

print("Table created and populated.")

在包含演示用户和已填充的示例表后，剩余的配置涉及设置嵌入和摘要功能。用户可以选择多种提供商，包括本地数据库解决方案和第三方服务，如 Ocigenai、Hugging Face 和 OpenAI。如果用户选择第三方提供商，则需要建立包含必要身份验证详细信息的凭据。反之，如果选择数据库作为嵌入提供商，则需要将 ONNX 模型上传到 Oracle Database。使用数据库选项时，摘要功能无需额外设置。

### 加载 ONNX 模型

Oracle 支持多种嵌入提供商，使您可以在专有数据库解决方案和第三方服务（如 Oracle Generative AI Service 和 HuggingFace）之间进行选择。此选择决定了生成和管理嵌入的方法。

***重要提示***：如果您选择数据库选项，则必须将 ONNX 模型上传到 Oracle 数据库。反之，如果选择第三方提供商来生成嵌入，则无需将 ONNX 模型上传到 Oracle 数据库。

直接在 Oracle 数据库中使用 ONNX 模型的一个显著优势是它提供了增强的安全性和性能，无需将数据传输到外部方。此外，此方法避免了通常与网络或 REST API 调用相关的延迟。

以下是将 ONNX 模型上传到 Oracle 数据库的示例代码：

In [ ]:
from langchain_community.embeddings.oracleai import OracleEmbeddings

# please update with your related information
# make sure that you have onnx file in the system
onnx_dir = "DEMO_PY_DIR"
onnx_file = "tinybert.onnx"
model_name = "demo_model"

OracleEmbeddings.load_onnx_model(connection, onnx_dir, onnx_file, model_name)
print("ONNX model loaded.")

### 创建凭证

在选择第三方提供商生成嵌入时，用户需要建立凭证以安全地访问提供商的端点。

***重要提示：*** 选择“数据库”提供商生成嵌入时，无需凭证。但是，如果用户决定使用第三方提供商，则必须创建特定于所选提供商的凭证。

下面是一个示例：

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        """
       declare
           jo json_object_t;
       begin
           -- HuggingFace
           dbms_vector_chain.drop_credential(credential_name  => 'HF_CRED');
           jo := json_object_t();
           jo.put('access_token', '<access_token>');
           dbms_vector_chain.create_credential(
               credential_name   =>  'HF_CRED',
               params            => json(jo.to_string));

           -- OCIGENAI
           dbms_vector_chain.drop_credential(credential_name  => 'OCI_CRED');
           jo := json_object_t();
           jo.put('user_ocid','<user_ocid>');
           jo.put('tenancy_ocid','<tenancy_ocid>');
           jo.put('compartment_ocid','<compartment_ocid>');
           jo.put('private_key','<private_key>');
           jo.put('fingerprint','<fingerprint>');
           dbms_vector_chain.create_credential(
               credential_name   => 'OCI_CRED',
               params            => json(jo.to_string));
       end;
       """
    )

### 加载文档
您可以通过相应地配置加载器参数，灵活地从 Oracle 数据库、文件系统或两者中加载文档。有关这些参数的详细信息，请参阅 [Oracle AI Vector Search Guide](https://docs.oracle.com/en/database/oracle/oracle-database/23/arpls/dbms_vector_chain1.html#GUID-73397E89-92FB-48ED-94BB-1AD960C4EA1F)。

使用 OracleDocLoader 的一个显著优势是它能够处理超过 150 种不同的文件格式，无需为不同文档类型使用多个加载器。支持格式的完整列表，请参阅 [Oracle Text Supported Document Formats](https://docs.oracle.com/en/database/oracle/oracle-database/23/ccref/oracle-text-supported-document-formats.html)。

下面是一个演示如何使用 OracleDocLoader 的示例代码片段：

In [ ]:
from langchain_community.document_loaders.oracleai import OracleDocLoader
from langchain_core.documents import Document

# loading from Oracle Database table
# make sure you have the table with this specification
loader_params = {
    "owner": "testuser",
    "tablename": "demo_tab",
    "colname": "data",
}

""" load the docs """
loader = OracleDocLoader(conn=connection, params=loader_params)
docs = loader.load()

""" verify """
print(f"Number of docs loaded: {len(docs)}")
# print(f"Document-0: {docs[0].page_content}") # content

### 生成摘要
加载文档后，您可能需要为每个文档生成摘要。Oracle AI Vector Search LangChain 库提供了一套专为文档摘要设计的 API。它支持多种摘要提供商，例如 Database、Oracle Generative AI Service、HuggingFace 等，允许您选择最适合您需求的提供商。要利用这些功能，您必须按照规定配置摘要参数。有关这些参数的详细信息，请参阅 [Oracle AI Vector Search 指南](https://docs.oracle.com/en/database/oracle/oracle-database/23/arpls/dbms_vector_chain1.html#GUID-EC9DDB58-6A15-4B36-BA66-ECBA20D2CE57)。

***注意：*** 如果您想使用 Oracle 内部和默认提供商 'database' 以外的第三方摘要生成提供商，您可能需要设置代理。如果您没有代理，请在实例化 OracleSummary 时移除 proxy 参数。

In [ ]:
# proxy to be used when we instantiate summary and embedder objects
proxy = ""

以下示例代码展示了如何生成摘要：

In [ ]:
from langchain_community.utilities.oracleai import OracleSummary
from langchain_core.documents import Document

# using 'database' provider
summary_params = {
    "provider": "database",
    "glevel": "S",
    "numParagraphs": 1,
    "language": "english",
}

# get the summary instance
# Remove proxy if not required
summ = OracleSummary(conn=connection, params=summary_params, proxy=proxy)

list_summary = []
for doc in docs:
    summary = summ.get_summary(doc.page_content)
    list_summary.append(summary)

""" verify """
print(f"Number of Summaries: {len(list_summary)}")
# print(f"Summary-0: {list_summary[0]}") #content

### 分割文档
文档的大小可能各不相同，从小文档到非常大的文档都有。用户通常倾向于将文档分割成更小的部分，以便于生成嵌入。此分割过程提供了广泛的自定义选项。有关这些参数的详细信息，请参阅 [Oracle AI Vector Search Guide](https://docs.oracle.com/en/database/oracle/oracle-database/23/arpls/dbms_vector_chain1.html#GUID-4E145629-7098-4C7C-804F-FC85D1F24240)。

下面是一个演示如何实现此功能的示例代码：

In [ ]:
from langchain_community.document_loaders.oracleai import OracleTextSplitter
from langchain_core.documents import Document

# split by default parameters
splitter_params = {"normalize": "all"}

""" get the splitter instance """
splitter = OracleTextSplitter(conn=connection, params=splitter_params)

list_chunks = []
for doc in docs:
    chunks = splitter.split_text(doc.page_content)
    list_chunks.extend(chunks)

""" verify """
print(f"Number of Chunks: {len(list_chunks)}")
# print(f"Chunk-0: {list_chunks[0]}") # content

### 生成嵌入

现在文档已按要求分块，您可能需要为这些块生成嵌入。Oracle AI Vector Search 提供了多种生成嵌入的方法，可以利用本地托管的 ONNX 模型或第三方 API。有关配置这些替代方案的全面说明，请参阅 [Oracle AI Vector Search 指南](https://docs.oracle.com/en/database/oracle/oracle-database/23/arpls/dbms_vector_chain1.html#GUID-C6439E94-4E86-4ECD-954E-4B73D53579DE)。

***注意：*** 您可能需要配置代理才能使用第三方嵌入生成提供商，但“database”提供商除外，它使用 ONNX 模型。

In [ ]:
# proxy to be used when we instantiate summary and embedder object
proxy = ""

以下示例代码展示了如何生成嵌入：

In [ ]:
from langchain_community.embeddings.oracleai import OracleEmbeddings
from langchain_core.documents import Document

# using ONNX model loaded to Oracle Database
embedder_params = {"provider": "database", "model": "demo_model"}

# get the embedding instance
# Remove proxy if not required
embedder = OracleEmbeddings(conn=connection, params=embedder_params, proxy=proxy)

embeddings = []
for doc in docs:
    chunks = splitter.split_text(doc.page_content)
    for chunk in chunks:
        embed = embedder.embed_query(chunk)
        embeddings.append(embed)

""" verify """
print(f"Number of embeddings: {len(embeddings)}")
# print(f"Embedding-0: {embeddings[0]}") # content

## 创建 Oracle AI 向量存储

既然您已了解如何单独使用 Oracle AI LangChain 库 API 来处理文档，接下来我们将展示如何与 Oracle AI 向量存储集成，以方便进行语义搜索。

首先，我们导入所有依赖项：

In [ ]:
import sys

import oracledb
from langchain_community.document_loaders.oracleai import (
    OracleDocLoader,
    OracleTextSplitter,
)
from langchain_community.embeddings.oracleai import OracleEmbeddings
from langchain_community.utilities.oracleai import OracleSummary
from langchain_community.vectorstores import oraclevs
from langchain_community.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document

接下来，我们将所有文档处理阶段组合在一起。以下是示例代码：

In [ ]:
"""
In this sample example, we will use 'database' provider for both summary and embeddings
so, we don't need to do the following:
    - set proxy for 3rd party providers
    - create credential for 3rd party providers

If you choose to use 3rd party provider, please follow the necessary steps for proxy and credential.
"""

# please update with your username, password, and database connection string
username = ""
password = ""
dsn = ""

with oracledb.connect(user=username, password=password, dsn=dsn) as connection:
    print("Connection successful!")

    # load onnx model
    # please update with your related information
    onnx_dir = "DEMO_PY_DIR"
    onnx_file = "tinybert.onnx"
    model_name = "demo_model"
    OracleEmbeddings.load_onnx_model(connection, onnx_dir, onnx_file, model_name)
    print("ONNX model loaded.")

    # params
    # please update necessary fields with related information
    loader_params = {
        "owner": "testuser",
        "tablename": "demo_tab",
        "colname": "data",
    }
    summary_params = {
        "provider": "database",
        "glevel": "S",
        "numParagraphs": 1,
        "language": "english",
    }
    splitter_params = {"normalize": "all"}
    embedder_params = {"provider": "database", "model": "demo_model"}

    # instantiate loader, summary, splitter, and embedder
    loader = OracleDocLoader(conn=connection, params=loader_params)
    summary = OracleSummary(conn=connection, params=summary_params)
    splitter = OracleTextSplitter(conn=connection, params=splitter_params)
    embedder = OracleEmbeddings(conn=connection, params=embedder_params)

    # process the documents
    chunks_with_mdata = []
    for id, doc in enumerate(docs, start=1):
        summ = summary.get_summary(doc.page_content)
        chunks = splitter.split_text(doc.page_content)
        for ic, chunk in enumerate(chunks, start=1):
            chunk_metadata = doc.metadata.copy()
            chunk_metadata["id"] = (
                chunk_metadata["_oid"] + "$" + str(id) + "$" + str(ic)
            )
            chunk_metadata["document_id"] = str(id)
            chunk_metadata["document_summary"] = str(summ[0])
            chunks_with_mdata.append(
                Document(page_content=str(chunk), metadata=chunk_metadata)
            )

    """ verify """
    print(f"Number of total chunks with metadata: {len(chunks_with_mdata)}")

至此，我们已处理完文档并生成了带有元数据的块。接下来，我们将使用这些块创建 Oracle AI Vector Store。

以下是执行此操作的示例代码：

In [ ]:
# create Oracle AI Vector Store
vectorstore = OracleVS.from_documents(
    chunks_with_mdata,
    embedder,
    client=connection,
    table_name="oravs",
    distance_strategy=DistanceStrategy.DOT_PRODUCT,
)

""" verify """
print(f"Vector Store Table: {vectorstore.table_name}")

提供的示例说明了使用 DOT_PRODUCT 距离策略创建向量存储。用户可以灵活地使用 Oracle AI Vector Store 的各种距离策略，具体详情请参阅我们的[综合指南](https://python.langchain.com/v0.1/docs/integrations/vectorstores/oracle/)。

在将嵌入存储在向量数据库后，建议建立索引以提高查询执行期间的语义搜索性能。

***注意*** 如果您遇到“内存不足”错误，建议在数据库配置中增加 ***vector_memory_size***。

以下是创建索引的示例代码片段：

In [ ]:
oraclevs.create_index(
    connection, vectorstore, params={"idx_name": "hnsw_oravs", "idx_type": "HNSW"}
)

print("Index created.")

此示例演示了在 `oravs` 表中的嵌入上创建默认 HNSW 索引。您可以根据自己的具体需求调整各种参数。有关这些参数的详细信息，请参阅 [Oracle AI Vector Search Guide book](https://docs.oracle.com/en/database/oracle/oracle-database/23/vecse/manage-different-categories-vector-indexes.html)。

此外，还可以创建各种类型的向量索引以满足不同的需求。更多详细信息可在我们的 [comprehensive guide](https://python.langchain.com/v0.1/docs/integrations/vectorstores/oracle/) 中找到。

## 执行语义搜索
一切就绪！

您已成功处理文档并将其存储在向量存储中，随后创建了索引以提高查询性能。现在，您可以进行语义搜索了。

以下是此过程的示例代码：

In [ ]:
query = "What is Oracle AI Vector Store?"
filter = {"document_id": ["1"]}

# Similarity search without a filter
print(vectorstore.similarity_search(query, 1))

# Similarity search with a filter
print(vectorstore.similarity_search(query, 1, filter=filter))

# Similarity search with relevance score
print(vectorstore.similarity_search_with_score(query, 1))

# Similarity search with relevance score with filter
print(vectorstore.similarity_search_with_score(query, 1, filter=filter))

# Max marginal relevance search
print(vectorstore.max_marginal_relevance_search(query, 1, fetch_k=20, lambda_mult=0.5))

# Max marginal relevance search with filter
print(
    vectorstore.max_marginal_relevance_search(
        query, 1, fetch_k=20, lambda_mult=0.5, filter=filter
    )
)